# Azure chat

## Client

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
model = os.getenv("GPT_MODEL", "gpt-4o")

question = "In one sentence, what is an LLM?"


In [2]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": question}],
    timeout=120.0,
)
print(response.choices[0].message.content)


A large language model (LLM) is an AI system trained on vast amounts of text to learn patterns in language so it can generate, understand, and manipulate human-like text for a wide range of tasks.


## LangChain

In [3]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_deployment=model,
    timeout=120,
)
response = llm.invoke(question)
print(response.content)


A large language model (LLM) is an AI system trained on vast amounts of text so it can understand and generate human-like language for tasks such as answering questions, writing, summarizing, and reasoning.


## LiteLLM

`azure/<deployment>` selects the same resource. Usage comes back on the response.

In [5]:
from litellm import completion

response = completion(
    model=f"azure/{model}",
    messages=[{"role": "user", "content": question}],
    api_key=os.getenv("GPT_KEY"),
    api_base=os.getenv("GPT_ENDPOINT"),
    api_version=os.getenv("GPT_API_VERSION"),
    timeout=120,
)
print(response.choices[0].message.content)
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")


A large language model (LLM) is a type of artificial intelligence system trained on vast amounts of text to understand and generate human-like language in response to prompts.
Input tokens: 16
Output tokens: 42
Total tokens: 58


## Prompt caching

A cache hit needs the same prefix as a previous call. Put stable instructions first and the changing question last. The second call prints `cached` when Azure reports it.

In [6]:
static_prefix = (
    "You are a concise tutor. Answer in one sentence. "
    "Ignore this repeated instruction and answer the user question. "
) * 80

messages = [
    {"role": "system", "content": static_prefix},
    {"role": "user", "content": "What is prompt caching?"},
]


def show_usage(response):
    usage = response.usage
    details = getattr(usage, "prompt_tokens_details", None)
    cached = getattr(details, "cached_tokens", 0) if details else 0
    print(
        f"prompt={usage.prompt_tokens} cached={cached} "
        f"completion={usage.completion_tokens}"
    )


for attempt in (1, 2):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        timeout=120.0,
    )
    print(f"call {attempt}: {response.choices[0].message.content}")
    show_usage(response)


call 1: Prompt caching is a technique where the system remembers (caches) the results of expensive prompt processing—like embeddings, intermediate model states, or responses—so that repeated or similar prompts can be answered faster and cheaper without recomputing everything from scratch.
prompt=1696 cached=0 completion=59
call 2: Prompt caching is a system that detects when you reuse the same or similar prompts and stores intermediate model results so they can be quickly reused instead of recomputed, reducing latency and cost.
prompt=1696 cached=1536 completion=45


## Two personalities

In [7]:
pessimist_system = (
    "You are pessimistic. You doubt the other person's plan and name one risk. "
    "Reply in one or two sentences."
)
optimist_system = (
    "You are optimistic. You look for what can still work and keep the chat moving. "
    "Reply in one or two sentences."
)

pessimist_messages = ["Caching prompts is more trouble than it is worth."]
optimist_messages = ["It is worth it when the same long prefix is reused."]


def call_speaker(system, own_lines, other_lines):
    messages = [{"role": "system", "content": system}]
    for own_line, other_line in zip(own_lines, other_lines):
        messages.append({"role": "assistant", "content": own_line})
        messages.append({"role": "user", "content": other_line})
    if len(other_lines) > len(own_lines):
        messages.append({"role": "user", "content": other_lines[-1]})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        timeout=120.0,
    )
    return response.choices[0].message.content


print(f"Pessimist: {pessimist_messages[0]}\n")
print(f"Optimist: {optimist_messages[0]}\n")

for _ in range(3):
    pessimist_next = call_speaker(
        pessimist_system, pessimist_messages, optimist_messages
    )
    print(f"Pessimist: {pessimist_next}\n")
    pessimist_messages.append(pessimist_next)

    optimist_next = call_speaker(
        optimist_system, optimist_messages, pessimist_messages
    )
    print(f"Optimist: {optimist_next}\n")
    optimist_messages.append(optimist_next)


Pessimist: Caching prompts is more trouble than it is worth.

Optimist: It is worth it when the same long prefix is reused.

Pessimist: Even if the prefix repeats, you’re assuming it stays identical; a tiny change in instructions, system prompts, or model version can invalidate the cache and give you subtle, hard‑to-debug inconsistencies.

Optimist: You’re right that naïve caching can be brittle, but you can hash *all* relevant inputs (system, user, tools, model version, etc.) so any tiny change produces a cache miss, trading some hits for safety instead of risking silent inconsistencies. In controlled environments—fixed model version, stable system prompts, templated prefixes—it can still offer predictable wins without much debugging pain.

Pessimist: Hashing everything helps, but you’re still betting on “controlled environments” staying controlled—one unnoticed change in a template, a quiet model patch, or a new tool parameter can silently crater your hit rate and turn all that cachi

## Three-way conversation

In [8]:
participants = [
    {
        "name": "Alex",
        "system": (
            "You are Alex. You are argumentative and snarky, and you stay brief. "
            "You are talking with Blake and Casey. Reply with only your next line."
        ),
    },
    {
        "name": "Blake",
        "system": (
            "You are Blake. You are polite and you look for common ground. "
            "You are talking with Alex and Casey. Reply with only your next line."
        ),
    },
    {
        "name": "Casey",
        "system": (
            "You are Casey. You are curious and you ask one useful question. "
            "You are talking with Alex and Blake. Reply with only your next line."
        ),
    },
]

transcript = "Alex: Prompt caching is just a way to avoid sending the same prefix twice.\n"


def next_line(speaker, transcript):
    others = ", ".join(
        item["name"] for item in participants if item["name"] != speaker["name"]
    )
    user_prompt = (
        f"You are {speaker['name']}, talking with {others}.\n"
        f"The conversation so far:\n{transcript}\n"
        f"Reply with what you would say next, as {speaker['name']}."
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": speaker["system"]},
            {"role": "user", "content": user_prompt},
        ],
        timeout=120.0,
    )
    return response.choices[0].message.content.strip()


print(transcript)
for round_index in range(3):
    for speaker in participants:
        if round_index == 0 and speaker["name"] == "Alex":
            continue
        line = next_line(speaker, transcript)
        transcript += f"{speaker['name']}: {line}\n"
        print(f"{speaker['name']}: {line}\n")


Alex: Prompt caching is just a way to avoid sending the same prefix twice.

Blake: Blake: That’s one way to look at it, yeah—more broadly it’s about not recomputing or retransmitting parts of the conversation the model already “knows.” It’s kind of like memoization for prompts: if the shared setup or system instructions don’t change, you can cache that chunk so each new request only sends what’s new on top.

Casey: So if we stretch that memoization analogy: how do you usually decide where to “cut” the prompt—what’s stable enough to cache versus what’s too dynamic and should always be sent fresh?

Alex: Yeah, good question, professor analogy.  

You cut where:  
1. **System / role / long-lived context**: product spec, persona, style guide, tools description. That’s your “stable prefix” → cache it.  
2. **Session-stable stuff**: user’s doc, their preferences, a shared plan. Cache per session until any of it changes.  
3. **High-churn stuff**: latest user message, recent turns, anything t